In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 8
plt.rcParams['hatch.linewidth'] = 0.5

def read_and_process_data(filepath, sheet_name):
    """Read LCA data and create separate dataframes for Low and High ginning scenarios"""
    df = pd.read_excel(filepath, sheet_name=sheet_name)

    # Find pro-electricity separator
    split_idx = None
    for idx, row in df.iterrows():
        if pd.notna(row.iloc[0]) and 'pro-electricity' in str(row.iloc[0]).lower():
            split_idx = idx
            break

    if split_idx is None:
        raise ValueError("Could not find 'pro-electricity' separator in data")

    # Split into baseline and pro-electricity
    base_df = df.iloc[:split_idx].dropna(subset=['State']).copy().reset_index(drop=True)
    pro_elec_df = df.iloc[split_idx+2:].dropna(subset=['State']).copy().reset_index(drop=True)

    return base_df, pro_elec_df

def separate_ginning_scenarios(df, category='GHG'):
    """Separate data into Low and High ginning scenarios"""

    # Define column patterns based on category
    if category == 'GHG':
        unit = '(kg CO2/1000kg lint)'
        ng_low = f'NG Ginning Low {unit}'
        ng_high = f'NG Ginning High {unit}'
        elec_low = f'Electricity Ginning Low {unit}'
        elec_high = f'Electricity Ginning High {unit}'
    elif category == 'Energy':
        unit = '(MJ/1000kg lint)'
        ng_low = f'NG Ginning Low {unit}'
        ng_high = f'NG Ginning High {unit}'
        elec_low = f'Electricity Ginning Low {unit}'
        elec_high = f'Electricity Ginning High {unit}'
    else:  # Water
        unit = '(L/1000kg lint)'
        ng_low = f'NG Ginning Low {unit}'
        ng_high = f'NG Ginning High {unit}'
        elec_low = f'Electricity Ginning Low {unit}'
        elec_high = f'Electricity Ginning High {unit}'

    # Get all columns except State and ginning columns
    state_col = df['State']
    ginning_cols = [ng_low, ng_high, elec_low, elec_high]
    other_cols = [col for col in df.columns if col not in ginning_cols + ['State']]

    # Create Low scenario dataframe - keep NG and Elec separate
    df_low = df[['State'] + other_cols].copy()
    df_low['NG Ginning'] = df[ng_low]
    df_low['Elec. Ginning'] = df[elec_low]
    df_low = df_low.set_index('State')

    # Create High scenario dataframe - keep NG and Elec separate
    df_high = df[['State'] + other_cols].copy()
    df_high['NG Ginning'] = df[ng_high]
    df_high['Elec. Ginning'] = df[elec_high]
    df_high = df_high.set_index('State')

    return df_low, df_high

def clean_column_names(df, category='GHG'):
    """Clean up column names for display"""
    rename_dict = {}

    for col in df.columns:
        if 'Herbicides' in col:
            rename_dict[col] = 'Herbicides'
        elif 'Insecticides' in col:
            rename_dict[col] = 'Insecticides'
        elif 'Phosphoric Acid' in col:
            rename_dict[col] = 'P₂O₅'
        elif 'Potassium Oxide' in col:
            rename_dict[col] = 'K₂O'
        elif 'Calcium Carbonate' in col:
            rename_dict[col] = 'CaCO₃'
        elif 'N Fertilizer' in col:
            rename_dict[col] = 'N Fertilizer'
        elif 'Diesel' in col:
            rename_dict[col] = 'Diesel'
        elif 'N2O Emission' in col or 'N₂O' in col:
            rename_dict[col] = 'N₂O Emission'
        elif 'Irrigation Water' in col:
            rename_dict[col] = 'Irrigation Water'
        elif 'Electricity' in col and 'Ginning' not in col:
            rename_dict[col] = 'Electricity'

    return df.rename(columns=rename_dict)

abbrev_list = ['AL', 'AZ', 'AR', 'CA', 'FL', 'GA', 'KS', 'LA', 'MS', 'MO', 'NM', 'NC', 'OK', 'SC', 'TN', 'TX', 'VA']

filepath = '../results/Cotton_LCA_17_States_0925.xlsx'

# ============================================================================
# CREATE MEGA PLOT - ALL THREE IMPACT CATEGORIES
# ============================================================================
print("Creating integrated mega plot...")

# Create figure with 12 subplots (3 rows x 4 columns)
fig = plt.figure(figsize=(7.2, 9.5))
gs = fig.add_gridspec(3, 4, hspace=0.4, wspace=0.3,
                      left=0.08, right=0.98, top=0.95, bottom=0.05)

# ============================================================================
# ROW 1: GHG EMISSIONS
# ============================================================================
base_df, pro_elec_df = read_and_process_data(filepath, 'Results_GHG_Breakdown')
base_low, base_high = separate_ginning_scenarios(base_df, 'GHG')
pro_low, pro_high = separate_ginning_scenarios(pro_elec_df, 'GHG')

base_low = clean_column_names(base_low, 'GHG')
base_high = clean_column_names(base_high, 'GHG')
pro_low = clean_column_names(pro_low, 'GHG')
pro_high = clean_column_names(pro_high, 'GHG')

ax_ghg = [fig.add_subplot(gs[0, 0])]
for i in range(1, 4):
    ax_ghg.append(fig.add_subplot(gs[0, i], sharey=ax_ghg[0]))

base_low.plot(kind='bar', stacked=True, ax=ax_ghg[0], legend=False, width=0.7)
ax_ghg[0].set_title('Baseline - Low', fontsize=8, pad=8)
ax_ghg[0].grid(True, axis='y', alpha=0.6)

base_high.plot(kind='bar', stacked=True, ax=ax_ghg[1], legend=False, width=0.7)
ax_ghg[1].set_title('Baseline - High', fontsize=8, pad=8)
ax_ghg[1].grid(True, axis='y', alpha=0.6)

pro_low.plot(kind='bar', stacked=True, ax=ax_ghg[2], legend=False, width=0.7)
ax_ghg[2].set_title('Pro-Electricity - Low', fontsize=8, pad=8)
ax_ghg[2].grid(True, axis='y', alpha=0.6)

pro_high.plot(kind='bar', stacked=True, ax=ax_ghg[3], legend=False, width=0.7)
ax_ghg[3].set_title('Pro-Electricity - High', fontsize=8, pad=8)
ax_ghg[3].grid(True, axis='y', alpha=0.6)

for idx, (df, ax) in enumerate([(base_low, ax_ghg[0]), (base_high, ax_ghg[1]),
                                  (pro_low, ax_ghg[2]), (pro_high, ax_ghg[3])]):
    totals = df.sum(axis=1)
    for i, (state, total) in enumerate(totals.items()):
        ax.text(i, total, f'{total:.0f}', ha='center', va='bottom', fontsize=5, rotation=90)

for i, ax in enumerate(ax_ghg):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xticklabels(abbrev_list, rotation=90, ha='center', fontsize=7)
    ax.set_xlabel('')
    if i > 0:
        ax.set_ylabel('')
        ax.tick_params(labelleft=False)

ax_ghg[0].set_ylabel('GHG Emissions\n(kg CO₂-eq/1000 kg lint)', fontsize=7)

ax_ghg[0].text(-0.15, 1.12, 'a', transform=ax_ghg[0].transAxes,
               fontsize=12, fontweight='bold', va='top')

# ============================================================================
# ROW 2: ENERGY USE
# ============================================================================
base_df, pro_elec_df = read_and_process_data(filepath, 'Results_Energy_Breakdown')
base_low, base_high = separate_ginning_scenarios(base_df, 'Energy')
pro_low, pro_high = separate_ginning_scenarios(pro_elec_df, 'Energy')

base_low = clean_column_names(base_low, 'Energy')
base_high = clean_column_names(base_high, 'Energy')
pro_low = clean_column_names(pro_low, 'Energy')
pro_high = clean_column_names(pro_high, 'Energy')

ax_energy = [fig.add_subplot(gs[1, 0])]
for i in range(1, 4):
    ax_energy.append(fig.add_subplot(gs[1, i], sharey=ax_energy[0]))

base_low.plot(kind='bar', stacked=True, ax=ax_energy[0], legend=False, width=0.7)
ax_energy[0].grid(True, axis='y', alpha=0.6)

base_high.plot(kind='bar', stacked=True, ax=ax_energy[1], legend=False, width=0.7)
ax_energy[1].grid(True, axis='y', alpha=0.6)

pro_low.plot(kind='bar', stacked=True, ax=ax_energy[2], legend=False, width=0.7)
ax_energy[2].grid(True, axis='y', alpha=0.6)

pro_high.plot(kind='bar', stacked=True, ax=ax_energy[3], legend=False, width=0.7)
ax_energy[3].grid(True, axis='y', alpha=0.6)

for idx, (df, ax) in enumerate([(base_low, ax_energy[0]), (base_high, ax_energy[1]),
                                  (pro_low, ax_energy[2]), (pro_high, ax_energy[3])]):
    totals = df.sum(axis=1)
    for i, (state, total) in enumerate(totals.items()):
        ax.text(i, total, f'{total:.0f}', ha='center', va='bottom', fontsize=5, rotation=90)

for i, ax in enumerate(ax_energy):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xticklabels(abbrev_list, rotation=90, ha='center', fontsize=7)
    ax.set_xlabel('')
    if i > 0:
        ax.set_ylabel('')
        ax.tick_params(labelleft=False)

ax_energy[0].set_ylabel('Energy Use\n(MJ/1000 kg lint)', fontsize=7)

ax_energy[0].text(-0.15, 1.12, 'b', transform=ax_energy[0].transAxes,
                  fontsize=12, fontweight='bold', va='top')

# ============================================================================
# ROW 3: WATER USE
# ============================================================================
base_df, pro_elec_df = read_and_process_data(filepath, 'Results_Water_Breakdown')
base_low, base_high = separate_ginning_scenarios(base_df, 'Water')
pro_low, pro_high = separate_ginning_scenarios(pro_elec_df, 'Water')

base_low = clean_column_names(base_low, 'Water')
base_high = clean_column_names(base_high, 'Water')
pro_low = clean_column_names(pro_low, 'Water')
pro_high = clean_column_names(pro_high, 'Water')

# Convert L to m³
base_low = base_low / 1000
base_high = base_high / 1000
pro_low = pro_low / 1000
pro_high = pro_high / 1000

ax_water = [fig.add_subplot(gs[2, 0])]
for i in range(1, 4):
    ax_water.append(fig.add_subplot(gs[2, i], sharey=ax_water[0]))

base_low.plot(kind='bar', stacked=True, ax=ax_water[0], legend=False, width=0.7)
ax_water[0].grid(True, axis='y', alpha=0.6)

base_high.plot(kind='bar', stacked=True, ax=ax_water[1], legend=False, width=0.7)
ax_water[1].grid(True, axis='y', alpha=0.6)

pro_low.plot(kind='bar', stacked=True, ax=ax_water[2], legend=False, width=0.7)
ax_water[2].grid(True, axis='y', alpha=0.6)

pro_high.plot(kind='bar', stacked=True, ax=ax_water[3], legend=False, width=0.7)
ax_water[3].grid(True, axis='y', alpha=0.6)

for idx, (df, ax) in enumerate([(base_low, ax_water[0]), (base_high, ax_water[1]),
                                  (pro_low, ax_water[2]), (pro_high, ax_water[3])]):
    totals = df.sum(axis=1)
    for i, (state, total) in enumerate(totals.items()):
        ax.text(i, total, f'{total:.1f}', ha='center', va='bottom', fontsize=5, rotation=90)

for i, ax in enumerate(ax_water):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xticklabels(abbrev_list, rotation=90, ha='center', fontsize=7)
    ax.set_xlabel('')
    if i > 0:
        ax.set_ylabel('')
        ax.tick_params(labelleft=False)

ax_water[0].set_ylabel('Water Use\n(m³/1000 kg lint)', fontsize=7)

ax_water[0].text(-0.15, 1.12, 'c', transform=ax_water[0].transAxes,
                 fontsize=12, fontweight='bold', va='top')

# ============================================================================
# CREATE SHARED LEGEND AT THE TOP
# ============================================================================
handles, labels = ax_ghg[0].get_legend_handles_labels()
fig.legend(handles, labels, bbox_to_anchor=(0.5, 1.05), loc='upper center', ncol=6,
           frameon=False, fontsize=6.5)

plt.savefig('../outputs/lca_17states/lca_state_all_impacts.tiff', dpi=300, bbox_inches='tight')
plt.close()



Creating integrated mega plot...
